In [1]:
import os
import torch
import numpy as np
from collections import defaultdict
import pprint
import pandas as pd
import sys
sys.path.append('../src')
from utils import round_to_perm


def summarize(values):
    return {
        'mean': np.nanmean(values),
        'sd': np.nanstd(values)
    }

B = 100 # Change as needed
n_i = 6 # Change as needed
result_folder = f'../data/results/B_{B}_n_{n_i}'
data_folder = f'../data/B_{B}_n_{n_i}'
seed = 1  # Example seed
file_path = os.path.join(data_folder, f'data_seed_{seed}.pt')
data = torch.load(file_path, map_location='cpu')

# Extract perm_matrix_x and perm_matrix_s
perm_matrix_x = data['perm_matrix_x']
perm_matrix_s = data['perm_matrix_s']

stats = {
    'GPmodel': defaultdict(list),
    'GPArealModel': defaultdict(list),
    'VIGP_unlinked': defaultdict(lambda: defaultdict(list))
}

for fname in os.listdir(result_folder):
    if not fname.startswith('results_seed_') or not fname.endswith('.pt'):
        continue
    path = os.path.join(result_folder, fname)
    result = torch.load(path, map_location='cpu')

    for model in ['GPmodel', 'GPArealModel']:
        if model in result:
            m = result[model]
            stats[model]['beta'].append(m.get('beta', [np.nan])[0])
            stats[model]['sigmasq'].append(m.get('sigmasq', np.nan))
            stats[model]['tausq'].append(m.get('tausq', np.nan))
            stats[model]['phi'].append(m.get('phi', np.nan))
            stats[model]['n_correct_perm_x'].append(np.nan)
            stats[model]['n_correct_perm_s'].append(np.nan)

    unique_tau_values = sorted(float(key.split('_')[-1]) for key in result if key.startswith('VIGP_unlinked_tau_'))

    for tau in unique_tau_values:
        key = f'VIGP_unlinked_tau_{tau}'
        if key in result:
            vi = result[key]
            beta = vi.get('mu_lambda_beta', np.nan)
            sig_beta = vi.get('sigmasq_lambda_beta', np.nan)

            a1, b1 = vi.get('lambda_a1', np.nan), vi.get('lambda_b1', np.nan)
            a2, b2 = vi.get('lambda_a2', np.nan), vi.get('lambda_b2', np.nan)

            sigmasq = b1 / (a1 - 1) if a1 > 1 else np.nan
            tausq = b2 / (a2 - 1) if a2 > 1 else np.nan

            sd_sigmasq = (b1**2) / ((a1 - 1)**2 * (a1 - 2)) if a1 > 2 else np.nan
            sd_tausq = (b2**2) / ((a2 - 1)**2 * (a2 - 2)) if a2 > 2 else np.nan

            phi = vi.get('mean_phi', np.nan)

            stats['VIGP_unlinked'][tau]['beta'].append(beta)
            stats['VIGP_unlinked'][tau]['sigmasq'].append(sigmasq)
            stats['VIGP_unlinked'][tau]['tausq'].append(tausq)
            stats['VIGP_unlinked'][tau]['phi'].append(phi)
            stats['VIGP_unlinked'][tau]['sd_sigmasq'].append(sd_sigmasq)
            stats['VIGP_unlinked'][tau]['sd_tausq'].append(sd_tausq)
            stats['VIGP_unlinked'][tau]['sd_beta'].append(np.sqrt(sig_beta))
            M_X_star = result[key]['M_X_star'].detach().numpy()
            M_S_star = result[key]['M_S_star'].detach().numpy()

            n_correct_perm_x = torch.sum(perm_matrix_x.T * round_to_perm(M_X_star))
            n_correct_perm_s = torch.sum(perm_matrix_s.T * round_to_perm(M_S_star))

            stats['VIGP_unlinked'][tau]['n_correct_perm_x'].append(n_correct_perm_x.item())
            stats['VIGP_unlinked'][tau]['n_correct_perm_s'].append(n_correct_perm_s.item())

summary = {
    model: {k: summarize(v) for k, v in stats[model].items()}
    for model in ['GPmodel', 'GPArealModel']
}
summary['VIGP_unlinked'] = {
    tau: {
        'beta': {'mean': np.nanmean(v['beta']), 'sd': np.nanmean(v['sd_beta'])},
        'sigmasq': {'mean': np.nanmean(v['sigmasq']), 'sd': np.nanmean(v['sd_sigmasq'])},
        'tausq': {'mean': np.nanmean(v['tausq']), 'sd': np.nanmean(v['sd_tausq'])},
        'phi': {'mean': np.nanmean(v['phi']), 'sd': np.nanstd(v['phi'])},
        'n_correct_perm_x': {'mean': np.nanmean(v['n_correct_perm_x']), 'sd': np.nanstd(v['n_correct_perm_x'])},
        'n_correct_perm_s': {'mean': np.nanmean(v['n_correct_perm_s']), 'sd': np.nanstd(v['n_correct_perm_s'])}
    } for tau, v in stats['VIGP_unlinked'].items()
}

# Prepare data for the table
data = {
    'GPmodel': [summary['GPmodel']['beta']['mean'], summary['GPmodel']['beta']['sd']],
    'GPArealModel': [summary['GPArealModel']['beta']['mean'], summary['GPArealModel']['beta']['sd']],
}

for tau in unique_tau_values:
    data[f'VI_tau_{tau}'] = [
        summary['VIGP_unlinked'][tau]['beta']['mean'],
        summary['VIGP_unlinked'][tau]['beta']['sd']
    ]

# Create a DataFrame for the table
table = pd.DataFrame(data, index=['Mean', 'SD'])

# Add rows for beta, phi, tausq, and sigmasq in "mean (sd)" format
data_extended = {
    'GPmodel': [
        f"{summary['GPmodel']['beta']['mean']:.4f} ({summary['GPmodel']['beta']['sd']:.4f})",
        f"{summary['GPmodel']['phi']['mean']:.4f} ({summary['GPmodel']['phi']['sd']:.4f})",
        f"{summary['GPmodel']['tausq']['mean']:.4f} ({summary['GPmodel']['tausq']['sd']:.4f})",
        f"{summary['GPmodel']['sigmasq']['mean']:.4f} ({summary['GPmodel']['sigmasq']['sd']:.4f})",
        f"{summary['GPmodel']['n_correct_perm_x']['mean']:.4f} ({summary['GPmodel']['n_correct_perm_x']['sd']:.4f})",
        f"{summary['GPmodel']['n_correct_perm_s']['mean']:.4f} ({summary['GPmodel']['n_correct_perm_s']['sd']:.4f})"
    ],
    'GPArealModel': [
        f"{summary['GPArealModel']['beta']['mean']:.4f} ({summary['GPArealModel']['beta']['sd']:.4f})",
        f"{summary['GPArealModel']['phi']['mean']:.4f} ({summary['GPArealModel']['phi']['sd']:.4f})",
        f"{summary['GPArealModel']['tausq']['mean']:.4f} ({summary['GPArealModel']['tausq']['sd']:.4f})",
        f"{summary['GPArealModel']['sigmasq']['mean']:.4f} ({summary['GPArealModel']['sigmasq']['sd']:.4f})",
        f"{summary['GPArealModel']['n_correct_perm_x']['mean']:.4f} ({summary['GPArealModel']['n_correct_perm_x']['sd']:.4f})",
        f"{summary['GPArealModel']['n_correct_perm_s']['mean']:.4f} ({summary['GPArealModel']['n_correct_perm_s']['sd']:.4f})"
    ],
}

for tau in unique_tau_values:
    data_extended[f'VI_tau_{tau}'] = [
        f"{summary['VIGP_unlinked'][tau]['beta']['mean']:.4f} ({summary['VIGP_unlinked'][tau]['beta']['sd']:.4f})",
        f"{summary['VIGP_unlinked'][tau]['phi']['mean']:.4f} ({summary['VIGP_unlinked'][tau]['phi']['sd']:.4f})",
        f"{summary['VIGP_unlinked'][tau]['tausq']['mean']:.4f} ({summary['VIGP_unlinked'][tau]['tausq']['sd']:.4f})",
        f"{summary['VIGP_unlinked'][tau]['sigmasq']['mean']:.4f} ({summary['VIGP_unlinked'][tau]['sigmasq']['sd']:.4f})",
        f"{summary['VIGP_unlinked'][tau]['n_correct_perm_x']['mean']:.4f} ({summary['VIGP_unlinked'][tau]['n_correct_perm_x']['sd']:.4f})",
        f"{summary['VIGP_unlinked'][tau]['n_correct_perm_s']['mean']:.4f} ({summary['VIGP_unlinked'][tau]['n_correct_perm_s']['sd']:.4f})"
    ]

# Create a DataFrame for the extended table
table_extended = pd.DataFrame(
    data_extended,
    index=['Beta', 'Phi', 'Tausq', 'Sigmasq', 'n_correct_perm_x', 'n_correct_perm_s']
)


# Display the updated table
print(f"B = {B}, and n = {n_i}.")
print(table_extended)

/var/folders/w7/jxz2zn316391355qstwl03940000gn/T/ipykernel_98486/2380706817.py:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location='c

B = 100, and n = 6.
                          GPmodel     GPArealModel      VI_tau_0.05  \
Beta              7.9283 (0.1269)  7.4714 (0.4398)  5.3313 (0.1545)   
Phi               4.8274 (0.9204)  3.9327 (1.2310)  5.1222 (1.4654)   
Tausq             0.2507 (0.0610)  0.9984 (0.4283)  4.8408 (0.0796)   
Sigmasq           4.2329 (0.5270)  4.2754 (0.7083)  3.4334 (0.1355)   
n_correct_perm_x        nan (nan)        nan (nan)  6.0000 (0.0000)   
n_correct_perm_s        nan (nan)        nan (nan)  5.6700 (0.8492)   

                       VI_tau_0.1       VI_tau_0.2       VI_tau_0.3  \
Beta              7.7346 (0.0922)  8.2857 (0.0681)  8.0534 (0.0671)   
Phi               5.0603 (1.4176)  5.0136 (1.2385)  5.0775 (1.1854)   
Tausq             1.7701 (0.0136)  0.9265 (0.0031)  0.9019 (0.0029)   
Sigmasq           2.5015 (0.1264)  2.7624 (0.0459)  3.1442 (0.0619)   
n_correct_perm_x  6.0000 (0.0000)  6.0000 (0.0000)  6.0000 (0.0000)   
n_correct_perm_s  5.9400 (0.5970)  5.9800 (0.1990)  6.00

/var/folders/w7/jxz2zn316391355qstwl03940000gn/T/ipykernel_98486/2380706817.py:14: RuntimeWarning: Mean of empty slice
  'mean': np.nanmean(values),
/opt/anaconda3/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [2]:
for tau in unique_tau_values:
    n_correct_perm_x_values = stats['VIGP_unlinked'][tau]['n_correct_perm_x']
    beta_values = stats['VIGP_unlinked'][tau]['beta']
    print(f"Tau: {tau}")
    print(f"{'n_correct_perm_x':<20}{'beta':<20}")
    for n_correct, beta_val in zip(n_correct_perm_x_values, beta_values):
        print(f"{n_correct:<20}{beta_val:<20}")
    print()

Tau: 0.2
n_correct_perm_x    beta                
6.0                 1.0682663917541504  
0.0                 -0.41068458557128906
6.0                 0.9541415572166443  
0.0                 -0.11679774522781372
6.0                 0.9218667149543762  
6.0                 0.8856166005134583  
0.0                 -0.2639894485473633 
6.0                 1.0336766242980957  
6.0                 0.9410961866378784  
0.0                 -0.19178389012813568

Tau: 0.4
n_correct_perm_x    beta                
6.0                 1.725868582725525   
0.0                 -0.1461365520954132 
6.0                 1.6856828927993774  
4.0                 0.2582753896713257  
6.0                 1.4643654823303223  
6.0                 1.909338355064392   
0.0                 -0.07066553086042404
6.0                 1.6975589990615845  
6.0                 2.0572235584259033  
0.0                 -0.036641258746385574

Tau: 0.6
n_correct_perm_x    beta                
6.0                 1.77872

In [ ]:
import matplotlib.pyplot as plt

for tau in unique_tau_values:  # Limit to first 5 plots
    plt.figure(figsize=(10, 6))

    for fname in os.listdir(result_folder):
        if not fname.startswith('results_seed_') or not fname.endswith('.pt'):
            continue
        path = os.path.join(result_folder, fname)
        result = torch.load(path, map_location='cpu')

        key = f'VIGP_unlinked_tau_{tau}'
        if key in result:
            loss_vector = result[key]['loss_vector']
            loss_array = loss_vector.detach().numpy()

            # Check if the loss vector is diverging (increasing)
        
            # Extract the seed number from the filename
            seed_number = fname.split('_')[-1].split('.')[0]

            # Plot the loss vector
            plt.plot(range(len(loss_array)), loss_array, linewidth=0.8)

            # Annotate the plot with the seed number
            plt.text(len(loss_array) - 1, loss_array[-1], seed_number, fontsize=8)

    # Add labels, legend, and title
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.title(f'Loss Vector for VIGP_unlinked_tau_{tau} Across Seeds')
    plt.legend()
    # Set y-axis to log scale
    plt.yscale('log')
    plt.show()


